# EEG-Based Alzheimer Detection — Deep Learning Pipeline
## Multi-Model Comparison: Multi-Scale CNN, CNN+Attention, LSTM, CNN+LSTM, CNN+BiLSTM+Attention, Transformer
### Dataset: HMMS.csv | Input: (samples, 250, 19) | Task: Binary Classification

In [ ]:
# ── Cell 1: Mount Drive & Download Dataset ──────────────────────────────────
!gdown --id 1GZyHxtjJbIVjT75yxcLiIgKeMd72KmcG
!ls -lh HMMS.csv

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1GZyHxtjJbIVjT75yxcLiIgKeMd72KmcG
From (redirected): https://drive.google.com/uc?id=1GZyHxtjJbIVjT75yxcLiIgKeMd72KmcG&confirm=t&uuid=f011d10f-10f8-4128-9134-360d6918f3ae
To: /content/HMMS.csv
100% 295M/295M [00:02<00:00, 98.5MB/s]
-rw-r--r-- 1 root root 282M Feb 22 08:21 HMMS.csv


In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ── Cell 3: Load & Segment Data ──────────────────────────────────────────────
data = pd.read_csv('HMMS.csv')
data = data.iloc[:, 1:]                          # remove index column
feature_cols = data.columns[:-1]
X_raw = data[feature_cols].iloc[:, :19].values   # 19 EEG channels
y_raw = data['label'].values

le = LabelEncoder()
y_raw = le.fit_transform(y_raw)
print('Classes:', le.classes_)
print('Raw EEG shape:', X_raw.shape)

# ── Segment into 1-second windows (fs=250) ───────────────────────────────────
fs = 250
window = 1 * fs  # 250 samples

segments, labels = [], []
for i in range(0, len(X_raw) - window, window):
    segments.append(X_raw[i:i + window])
    labels.append(y_raw[i])

X = np.array(segments, dtype=np.float32)   # (N, 250, 19)
y = np.array(labels,   dtype=np.int32)

print('Segmented shape:', X.shape)
print('Labels shape   :', y.shape)
print('Class distribution:', np.bincount(y))

Classes: ['Healthy' 'Mild' 'Moderate' 'Sever']
Raw EEG shape: (473552, 19)
Segmented shape: (1894, 250, 19)
Labels shape   : (1894,)
Class distribution: [ 95 480 720 599]


In [ ]:
# ── Cell 4: Normalise per-segment ────────────────────────────────────────────
# Z-score each channel independently across the time dimension
mean = X.mean(axis=1, keepdims=True)   # (N,1,19)
std  = X.std(axis=1,  keepdims=True) + 1e-8
X    = (X - mean) / std
print('After normalisation – mean≈0:', X.mean().round(4), ' std≈1:', X.std().round(4))

After normalisation – mean≈0: 0.0785  std≈1: 0.9867


In [ ]:
# ── Cell 5: Shared Attention Blocks ─────────────────────────────────────────

# ── 5a: Channel Attention (Squeeze-and-Excitation) ──────────────────────────
def channel_attention(x, ratio=8):
    """
    SE-style channel attention applied to (batch, time, channels).
    Learns which of the 19 EEG channels are most relevant.
    """
    c = x.shape[-1]
    # Global average pooling over time
    gap = layers.GlobalAveragePooling1D()(x)          # (B, C)
    se  = layers.Dense(max(c // ratio, 1), activation='relu')(gap)
    se  = layers.Dense(c, activation='sigmoid')(se)   # (B, C)
    se  = layers.Reshape((1, c))(se)                  # (B, 1, C)
    return layers.Multiply()([x, se])


# ── 5b: Temporal Attention ───────────────────────────────────────────────────
def temporal_attention(x):
    """
    Soft-attention over time steps.
    Learns which EEG time-steps carry diagnostic information.
    """
    # score per timestep
    score = layers.Dense(1, activation='tanh')(x)      # (B, T, 1)
    score = layers.Softmax(axis=1)(score)              # (B, T, 1)
    return layers.Multiply()([x, score])               # (B, T, C)


# ── 5c: Positional Encoding for Transformer ──────────────────────────────────
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len=250, d_model=64, **kwargs):
        super().__init__(**kwargs)
        self.max_len  = max_len
        self.d_model  = d_model

    def build(self, input_shape):
        positions = np.arange(self.max_len)[:, np.newaxis]
        dims      = np.arange(self.d_model)[np.newaxis, :]
        angles    = positions / np.power(10000, (2 * (dims // 2)) / self.d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.pe = tf.cast(angles[np.newaxis, :, :], tf.float32)  # (1, T, D)
        super().build(input_shape)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'max_len': self.max_len, 'd_model': self.d_model})
        return cfg


print('Attention blocks defined.')

Attention blocks defined.


In [ ]:
# ── Cell 6: Model 1 — Multi-Scale 1D CNN ─────────────────────────────────────
"""
Parallel CNN branches with kernel sizes that mimic EEG band frequency ranges:
  k=3  → Gamma  (≈high-freq)
  k=7  → Beta
  k=15 → Alpha
  k=31 → Theta
  k=63 → Delta  (≈low-freq)
Learnable filters replace traditional bandpass filtering.
"""

def build_multiscale_cnn(input_shape=(250, 19), n_classes=1):
    inp = Input(shape=input_shape, name='eeg_input')

    branches = []
    for k in [3, 7, 15, 31, 63]:                          # 5 frequency-band proxies
        b = layers.Conv1D(32, k, padding='same', use_bias=False)(inp)
        b = layers.BatchNormalization()(b)
        b = layers.Activation('relu')(b)
        b = layers.Conv1D(64, k, padding='same', use_bias=False)(b)
        b = layers.BatchNormalization()(b)
        b = layers.Activation('relu')(b)
        b = layers.MaxPooling1D(4)(b)
        b = layers.Dropout(0.3)(b)
        branches.append(b)

    x = layers.Concatenate()(branches)                    # merge all bands
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='sigmoid', name='output')(x)

    return Model(inp, out, name='MultiScale_CNN')


m = build_multiscale_cnn()
m.summary()

Model: "MultiScale_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 250, 19)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 250, 32)   │      1,824 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 250, 32)   │      4,256 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 250, 32)   │      9,120 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 250, 32)   │     18,848 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 250, 32)   │     38,304 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 250, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 32)   │        128 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 32)   │        128 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 32)   │        128 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 32)   │        128 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 250, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 250, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 250, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 250, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 250, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 250, 64)   │      6,144 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 250, 64)   │     14,336 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 250, 64)   │     30,720 │ activation_4[0][… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 458,145 (1.75 MB)

 Trainable params: 456,929 (1.74 MB)

 Non-trainable params: 1,216 (4.75 KB)

In [ ]:
# ── Cell 7: Model 2 — CNN + Channel & Temporal Attention ─────────────────────

def build_cnn_attention(input_shape=(250, 19), n_classes=1):
    inp = Input(shape=input_shape, name='eeg_input')

    # Multi-scale feature extraction
    branches = []
    for k in [3, 7, 15, 31]:
        b = layers.Conv1D(64, k, padding='same', use_bias=False)(inp)
        b = layers.BatchNormalization()(b)
        b = layers.Activation('relu')(b)
        branches.append(b)
    x = layers.Concatenate()(branches)

    x = layers.Conv1D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Channel Attention (SE Block)
    x = channel_attention(x, ratio=8)

    x = layers.Conv1D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(4)(x)

    # Temporal Attention
    x = temporal_attention(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='sigmoid', name='output')(x)

    return Model(inp, out, name='CNN_Attention')


m = build_cnn_attention()
m.summary()

Model: "CNN_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 250, 19)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 250, 64)   │      3,648 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 250, 64)   │      8,512 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 250, 64)   │     18,240 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 250, 64)   │     37,696 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_11[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_12[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_13[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_14[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_11       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_12       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_13       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 250, 256)  │          0 │ activation_10[0]… │
│ (Concatenate)       │                   │            │ activation_11[0]… │
│                     │                   │            │ activation_12[0]… │
│                     │                   │            │ activation_13[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 250, 128)  │     98,304 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 128)  │        512 │ conv1d_15[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_14       │ (None, 250, 128)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ activation_14[0]

 Total params: 238,610 (932.07 KB)

 Trainable params: 237,586 (928.07 KB)

 Non-trainable params: 1,024 (4.00 KB)

In [ ]:
# ── Cell 8: Model 3 — Stacked LSTM ───────────────────────────────────────────

def build_stacked_lstm(input_shape=(250, 19), n_classes=1):
    inp = Input(shape=input_shape, name='eeg_input')

    x = layers.LSTM(128, return_sequences=True,
                    dropout=0.3, recurrent_dropout=0.2)(inp)
    x = layers.LSTM(128, return_sequences=True,
                    dropout=0.3, recurrent_dropout=0.2)(x)
    x = layers.LSTM(64,  return_sequences=False,
                    dropout=0.3, recurrent_dropout=0.2)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='sigmoid', name='output')(x)

    return Model(inp, out, name='Stacked_LSTM')


m = build_stacked_lstm()
m.summary()

Model: "Stacked_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ eeg_input (InputLayer)          │ (None, 250, 19)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 250, 128)       │        75,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 250, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 260,993 (1019.50 KB)

 Trainable params: 260,993 (1019.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ── Cell 9: Model 4 — CNN + LSTM Hybrid ──────────────────────────────────────

def build_cnn_lstm(input_shape=(250, 19), n_classes=1):
    inp = Input(shape=input_shape, name='eeg_input')

    # CNN spatial feature extractor (per-channel patterns)
    x = layers.Conv1D(64, 7, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv1D(128, 5, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)      # (B, 125, 128)
    x = layers.Dropout(0.3)(x)

    # Channel Attention before LSTM
    x = channel_attention(x, ratio=8)

    # LSTM temporal modelling
    x = layers.LSTM(128, return_sequences=True,  dropout=0.3)(x)
    x = layers.LSTM(64,  return_sequences=False, dropout=0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='sigmoid', name='output')(x)

    return Model(inp, out, name='CNN_LSTM')


m = build_cnn_lstm()
m.summary()

Model: "CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 250, 19)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 250, 64)   │      8,512 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_17[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_18 (Conv1D)  │ (None, 250, 128)  │     40,960 │ activation_16[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 128)  │        512 │ conv1d_18[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_17       │ (None, 250, 128)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_6     │ (None, 125, 128)  │          0 │ activation_17[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 125, 128)  │          0 │ max_pooling1d_6[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ dropout_8[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 16)        │      2,064 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │      2,176 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 128)    │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 125, 128)  │          0 │ dropout_8[0][0],  │
│ (Multiply)          │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 125, 128)  │    131,584 │ multiply_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 64)        │     49,408 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128)       │      8,320 │ lstm_4[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 128)       │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │        129 │ dropout_9[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 243,921 (952.82 KB)

 Trainable params: 243,537 (951.32 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ── Cell 10: Model 5 — CNN + BiLSTM + Attention ───────────────────────────────

def build_cnn_bilstm_attention(input_shape=(250, 19), n_classes=1):
    inp = Input(shape=input_shape, name='eeg_input')

    # Multi-scale CNN backbone
    branches = []
    for k in [3, 7, 15, 31]:
        b = layers.Conv1D(64, k, padding='same', use_bias=False)(inp)
        b = layers.BatchNormalization()(b)
        b = layers.Activation('relu')(b)
        branches.append(b)
    x = layers.Concatenate()(branches)

    x = layers.Conv1D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # SE Channel Attention
    x = channel_attention(x, ratio=8)

    x = layers.MaxPooling1D(2)(x)   # (B, 125, 128)
    x = layers.Dropout(0.3)(x)

    # Bidirectional LSTM — captures both past and future context
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3))(x)
    x = layers.Bidirectional(layers.LSTM(64,  return_sequences=True, dropout=0.3))(x)

    # Temporal Attention over BiLSTM outputs
    x = temporal_attention(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='sigmoid', name='output')(x)

    return Model(inp, out, name='CNN_BiLSTM_Attention')


m = build_cnn_bilstm_attention()
m.summary()

Model: "CNN_BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 250, 19)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_19 (Conv1D)  │ (None, 250, 64)   │      3,648 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_20 (Conv1D)  │ (None, 250, 64)   │      8,512 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_21 (Conv1D)  │ (None, 250, 64)   │     18,240 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_22 (Conv1D)  │ (None, 250, 64)   │     37,696 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_19[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_20[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_21[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 64)   │        256 │ conv1d_22[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_18       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_19       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_20       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_21       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 250, 256)  │          0 │ activation_18[0]… │
│ (Concatenate)       │                   │            │ activation_19[0]… │
│                     │                   │            │ activation_20[0]… │
│                     │                   │            │ activation_21[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_23 (Conv1D)  │ (None, 250, 128)  │     98,304 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 128)  │        512 │ conv1d_23[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_22       │ (None, 250, 128)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ activation_22[0]

 Total params: 616,466 (2.35 MB)

 Trainable params: 615,698 (2.35 MB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
# ── Cell 11: Model 6 — Lightweight EEG Transformer ───────────────────────────

def transformer_encoder_block(x, d_model, num_heads, ff_dim, dropout_rate=0.1):
    """Single Transformer encoder block with pre-norm."""
    # Multi-Head Self-Attention
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout_rate
    )(x, x)                                       # self-attention
    attn_out = layers.Dropout(dropout_rate)(attn_out)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_out)  # residual + norm

    # Feed-forward
    ff = layers.Dense(ff_dim, activation='gelu')(x)
    ff = layers.Dropout(dropout_rate)(ff)
    ff = layers.Dense(d_model)(ff)
    ff = layers.Dropout(dropout_rate)(ff)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff)        # residual + norm
    return x


def build_eeg_transformer(input_shape=(250, 19), n_classes=1,
                           d_model=64, num_heads=4, ff_dim=128,
                           num_blocks=3, dropout_rate=0.2):
    inp = Input(shape=input_shape, name='eeg_input')

    # Patch / token embedding: project 19 channels → d_model
    x = layers.Conv1D(d_model, 1, padding='same')(inp)   # (B, 250, d_model)

    # Positional Encoding
    x = PositionalEncoding(max_len=250, d_model=d_model)(x)
    x = layers.Dropout(dropout_rate)(x)

    # Stacked Transformer Encoder Blocks
    for _ in range(num_blocks):
        x = transformer_encoder_block(
            x, d_model=d_model, num_heads=num_heads,
            ff_dim=ff_dim, dropout_rate=dropout_rate
        )

    # Temporal Attention-weighted pooling
    x = temporal_attention(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation='gelu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='sigmoid', name='output')(x)

    return Model(inp, out, name='EEG_Transformer')


m = build_eeg_transformer()
m.summary()

Model: "EEG_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 250, 19)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_24 (Conv1D)  │ (None, 250, 64)   │      1,280 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 250, 64)   │          0 │ conv1d_24[0][0]   │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 250, 64)   │          0 │ positional_encod… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 64)   │     16,640 │ dropout_12[0][0], │
│ (MultiHeadAttentio… │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 250, 64)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 250, 64)   │          0 │ dropout_12[0][0], │
│                     │                   │            │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 250, 64)   │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 250, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 250, 128)  │          0 │ dense_13[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 250, 64)   │      8,256 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 250, 64)   │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 250, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 64)   │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 64)   │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 250, 64)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 250, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 64)   │        128 │ add_2[0][0]     

 Total params: 110,210 (430.51 KB)

 Trainable params: 110,210 (430.51 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ── Cell 12: Training Utilities ───────────────────────────────────────────────

EPOCHS    = 50
BATCH     = 64
N_FOLDS   = 5
LR        = 1e-3

MODEL_BUILDERS = {
    'MultiScale_CNN'        : build_multiscale_cnn,
    'CNN_Attention'         : build_cnn_attention,
    'Stacked_LSTM'          : build_stacked_lstm,
    'CNN_LSTM'              : build_cnn_lstm,
    'CNN_BiLSTM_Attention'  : build_cnn_bilstm_attention,
    'EEG_Transformer'       : build_eeg_transformer,
}


def get_callbacks(model_name, fold):
    return [
        EarlyStopping(monitor='val_loss', patience=7,
                      restore_best_weights=True, verbose=0),
        ModelCheckpoint(
            f'best_{model_name}_fold{fold}.keras',
            monitor='val_loss', save_best_only=True, verbose=0
        ),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-6, verbose=0),
    ]


def compile_model(model):
    model.compile(
        optimizer=Adam(LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


print('Training config ready.')

Training config ready.


In [ ]:
# ── Cell 13: Training Loop (Stratified K-Fold) ────────────────────────────────

skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
results = []          # store per-model metrics
histories = {}        # store loss curves  {model_name: [fold_histories]}
cm_data   = {}        # store confusion matrices {model_name: conf_matrix}

for model_name, builder in MODEL_BUILDERS.items():
    print(f'\n{"="*60}')
    print(f'  Training: {model_name}')
    print(f'{"="*60}')

    fold_accs, fold_precs, fold_recs, fold_f1s = [], [], [], []
    fold_histories = []
    all_y_true, all_y_pred = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = builder(input_shape=(250, 19), n_classes=1)
        model = compile_model(model)

        hist = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH,
            callbacks=get_callbacks(model_name, fold),
            verbose=0
        )
        fold_histories.append(hist.history)

        # Predict
        y_prob = model.predict(X_val, verbose=0).flatten()
        y_pred = (y_prob >= 0.5).astype(int)

        acc  = accuracy_score(y_val, y_pred)
        prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_val, y_pred,    average='weighted', zero_division=0)
        f1   = f1_score(y_val, y_pred,        average='weighted', zero_division=0)

        fold_accs.append(acc)
        fold_precs.append(prec)
        fold_recs.append(rec)
        fold_f1s.append(f1)
        all_y_true.extend(y_val)
        all_y_pred.extend(y_pred)

        print(f'  Fold {fold}/{N_FOLDS}  '
              f'Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}')
        keras.backend.clear_session()   # free memory between folds

    # Aggregate
    histories[model_name] = fold_histories
    cm_data[model_name]   = confusion_matrix(all_y_true, all_y_pred)

    results.append({
        'Model'    : model_name,
        'Accuracy' : np.mean(fold_accs),
        'Precision': np.mean(fold_precs),
        'Recall'   : np.mean(fold_recs),
        'F1-Score' : np.mean(fold_f1s),
        'Acc_Std'  : np.std(fold_accs),
    })

    print(f'\n  ► Mean Acc={np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}')


# Build results DataFrame
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
results_df.to_csv('DL_EEG_results.csv', index=False)
print('\n\nFinal Results:')
print(results_df.to_string(index=False))


  Training: MultiScale_CNN
  Fold 1/5  Acc=0.2559  Prec=0.0857  Rec=0.2559  F1=0.1244
  Fold 2/5  Acc=0.2533  Prec=0.0642  Rec=0.2533  F1=0.1024
  Fold 3/5  Acc=0.2744  Prec=0.1157  Rec=0.2744  F1=0.1338
  Fold 4/5  Acc=0.2533  Prec=0.0642  Rec=0.2533  F1=0.1024
  Fold 5/5  Acc=0.1720  Prec=0.0524  Rec=0.1720  F1=0.0803

  ► Mean Acc=0.2418 ± 0.0358

  Training: CNN_Attention
  Fold 1/5  Acc=0.2533  Prec=0.0642  Rec=0.2533  F1=0.1024
  Fold 2/5  Acc=0.2533  Prec=0.0642  Rec=0.2533  F1=0.1024
  Fold 3/5  Acc=0.2533  Prec=0.0642  Rec=0.2533  F1=0.1024
  Fold 4/5  Acc=0.2533  Prec=0.0642  Rec=0.2533  F1=0.1024
  Fold 5/5  Acc=0.2540  Prec=0.0645  Rec=0.2540  F1=0.1029

  ► Mean Acc=0.2534 ± 0.0003

  Training: Stacked_LSTM


In [ ]:
# ── Cell 14: Visualisation 1 — Accuracy Comparison Bar Chart ─────────────────

plt.figure(figsize=(12, 6))
bars = plt.bar(
    results_df['Model'],
    results_df['Accuracy'] * 100,
    color=sns.color_palette('viridis', len(results_df)),
    edgecolor='black', linewidth=0.6
)
plt.errorbar(
    range(len(results_df)),
    results_df['Accuracy'] * 100,
    yerr=results_df['Acc_Std'] * 100,
    fmt='none', c='red', capsize=5, linewidth=1.5
)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
             f'{h:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.ylabel('Mean Accuracy (%)', fontsize=12)
plt.xlabel('Model', fontsize=12)
plt.title('Deep Learning Model Accuracy Comparison\n(EEG Alzheimer Detection)', fontsize=14)
plt.xticks(rotation=25, ha='right')
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 15: Visualisation 2 — Training vs Validation Loss Curves ────────────

n_models = len(MODEL_BUILDERS)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (model_name, folds) in zip(axes, histories.items()):
    # Average across folds
    max_ep = min(len(f['loss']) for f in folds)
    train_loss = np.mean([f['loss'][:max_ep]     for f in folds], axis=0)
    val_loss   = np.mean([f['val_loss'][:max_ep] for f in folds], axis=0)
    train_acc  = np.mean([f['accuracy'][:max_ep]     for f in folds], axis=0)
    val_acc    = np.mean([f['val_accuracy'][:max_ep] for f in folds], axis=0)

    epochs_range = range(1, max_ep + 1)
    ax.plot(epochs_range, train_loss, 'b-',  label='Train Loss',  linewidth=1.5)
    ax.plot(epochs_range, val_loss,   'r--', label='Val Loss',    linewidth=1.5)
    ax2 = ax.twinx()
    ax2.plot(epochs_range, train_acc * 100, 'g-',  label='Train Acc', linewidth=1, alpha=0.6)
    ax2.plot(epochs_range, val_acc   * 100, 'm--', label='Val Acc',   linewidth=1, alpha=0.6)
    ax2.set_ylabel('Accuracy (%)', color='g')

    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)

plt.suptitle('Training vs Validation Loss & Accuracy Curves\n(Mean over K-Folds)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 16: Visualisation 3 — Confusion Matrices ────────────────────────────

class_names = [str(c) for c in le.classes_]
n_models    = len(cm_data)
fig, axes   = plt.subplots(2, 3, figsize=(18, 11))
axes        = axes.flatten()

for ax, (model_name, cm) in zip(axes, cm_data.items()):
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2%', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.5, cbar=True
    )
    ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Normalised Confusion Matrices — All Models', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 17: Visualisation 4 — Performance Heatmap ───────────────────────────

metrics_df = results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
metrics_df = (metrics_df * 100).round(2)

plt.figure(figsize=(10, 6))
sns.heatmap(
    metrics_df,
    annot=True, fmt='.2f', cmap='YlGnBu',
    linewidths=0.5, linecolor='grey',
    vmin=50, vmax=100
)
plt.title('Model × Metric Performance Heatmap (%)',
          fontsize=14, fontweight='bold')
plt.ylabel('Model')
plt.xlabel('Metric')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('performance_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 18: Visualisation 5 — Multi-Metric Grouped Bar Chart ────────────────

plot_df  = results_df.set_index('Model')[['Accuracy','Precision','Recall','F1-Score']] * 100
x        = np.arange(len(plot_df))
width    = 0.20
colours  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, colour) in enumerate(zip(plot_df.columns, colours)):
    bars = ax.bar(x + i * width, plot_df[metric], width,
                  label=metric, color=colour, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}', ha='center', va='bottom',
                fontsize=7, rotation=90)

ax.set_ylabel('Score (%)')
ax.set_title('All Models — Accuracy / Precision / Recall / F1-Score',
             fontsize=13, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(plot_df.index, rotation=20, ha='right')
ax.legend()
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('multi_metric_bar.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 19: Final Performance Table ─────────────────────────────────────────

display_df = results_df[['Model','Accuracy','Precision','Recall','F1-Score','Acc_Std']].copy()
for col in ['Accuracy','Precision','Recall','F1-Score','Acc_Std']:
    display_df[col] = (display_df[col] * 100).round(2).astype(str) + ' %'

print('\n' + '='*72)
print('      FINAL DEEP LEARNING MODEL COMPARISON — EEG ALZHEIMER DETECTION')
print('='*72)
print(display_df.to_string(index=False))
print('='*72)
print(f'\nBest Model: {results_df.iloc[0]["Model"]}  '
      f'(Acc = {results_df.iloc[0]["Accuracy"]*100:.2f} %)')